# Fonctions communes d’analyse Bluesky

Adaptation de `../../bluesky_topics.ipynb` à la base SQLite et au prétraitement existants. Les fonctions sont chargées par les notebooks 04 à 07. Aucune collecte, aucun entraînement et aucun téléchargement n’est lancé lors du chargement de ces définitions.

Méthodes conservées : préparation du corpus, TF-IDF/K-Means, BERTopic avec embeddings/UMAP/HDBSCAN/MMR, séries temporelles, score d’émergence et regroupement Jaccard, comparaison aux médias.

In [ ]:
from pathlib import Path
import hashlib
import json
import re
import unicodedata
from functools import lru_cache

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import normalize

SEED = 42
EMB_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
TOKEN_PATTERN = r"(?u)\b[a-z]{3,}\b"


TOKEN_PATTERN = r"(?u)\b[a-z]{3,}\b"
LYON_PATTERN = re.compile(
    r"\b(?:lyon(?:nais|naise|naises)?|villeurbanne|venissieux|vaulx|bron|caluire|rhone|"
    r"part[ -]dieu|bellecour|croix[ -]rousse|confluence|perrache|guillotiere|tcl|oullins|"
    r"decines|saint[ -]fons|givors|tassin|ecully|meyzieu)\b", re.I
)


def remove_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

@lru_cache(maxsize=1)
def source_stopwords():
    """Mêmes listes NLTK française et personnalisée que le notebook source."""
    import nltk
    from nltk.corpus import stopwords
    cwd = Path.cwd()
    candidates = [cwd, cwd.parent, cwd / "bluesky-news-mining", cwd / "textmining" / "bluesky-news-mining"]
    root = next((p for p in candidates if (p / "notebooks/00_fonctions_base.ipynb").is_file()), None)
    if root is not None:
        local = str(root / "data/nltk_data")
        if local not in nltk.data.path:
            nltk.data.path.insert(0, local)
    try:
        raw_nltk = stopwords.words("french")
    except LookupError as error:
        raise LookupError("Installer les mots vides : python -m nltk.downloader -d data/nltk_data stopwords") from error
    custom = {'via', 'bien', 'quand', 'ete', 'tres', 'lol', 'comme', 'com', 'tout', 'jai', 'www', 'voir', 'veut', 'fait', 'plus', '2026', 'dune', 'encore', 'ptdr', 'lyonnaise', 'novembre', 'url', 'lyonnais', 'est', 'aussi', 'lyon', 'ont', 'fr', 'bsky', 'aout', 'sont', 'septembre', 'juste', 'octobre', 'amp', 'peut', 'html', 'dun', 'faire', 'faut', 'http', '2025', 'ca', 'france', 'bluesky', 'cette', 'nest', 'mdr', 'tous', 'cest', 'https', 'social', 'etre', 'alors'}
    noacc = {remove_accents(w) for w in raw_nltk} | custom
    accented = set(raw_nltk) | custom | noacc
    return noacc, accented


def analysis_text(text, lyon_only=False):
    STOP_NOACC, _ = source_stopwords()
    t = remove_accents(str(text).lower())
    t = re.sub(r"http\S+|www\S+", " ", t)
    t = re.sub(r"@[\w\.\-]+", " ", t)
    t = re.sub(r"['’\-–—]", " ", t)
    t = re.sub(r"[^\w\s]", " ", t)
    return " ".join(w for w in t.split() if w not in STOP_NOACC and not w.isdigit() and len(w) >= 3)

def prepare_analysis_corpus(posts, include_replies=False, lyon_only=False, min_words=5,
                            max_posts_per_author_day=5, exclude_authors=(),
                            auto_exclude_prolific=False, min_lexical_words=3):
    """Entrée : posts déjà nettoyés/dédoublonnés avec preprocess_posts."""
    required = {"uri", "did", "created_at", "text", "clean_text", "is_reply", "has_link"}
    if not required.issubset(posts.columns):
        raise ValueError(f"Colonnes manquantes : {sorted(required - set(posts.columns))}")
    if min_words < 1 or max_posts_per_author_day < 1 or min_lexical_words < 1:
        raise ValueError("Les seuils de longueur et de plafonnement doivent être positifs.")
    df = posts.copy()
    counts = [("Après prétraitement existant", len(df))]
    df["created_at"] = pd.to_datetime(df["created_at"], utc=True, errors="coerce", format="mixed")
    df = df.dropna(subset=["created_at"])
    df["date"] = df["created_at"].dt.tz_convert("Europe/Paris").dt.tz_localize(None).dt.normalize()
    # Pas de faux handle ou de faux compteurs : l'identifiant disponible est le DID.
    df["author"] = df["did"]
    df["author_did"] = df["did"]
    df["is_lyon"] = df["text"].map(lambda t: bool(LYON_PATTERN.search(remove_accents(t))))
    stats = df.groupby("author").agg(posts=("uri", "size"), link_share=("has_link", "mean"),
                                     duplicate_share=("text", lambda s: 1 - s.nunique() / len(s)))
    stats = stats.sort_values("posts", ascending=False)
    stats["suspect_activity"] = (stats["posts"] >= 50) & (
        (stats["link_share"] >= 0.5) | (stats["duplicate_share"] >= 0.3))
    excluded = set(exclude_authors)
    if auto_exclude_prolific:
        excluded |= set(stats.index[stats["suspect_activity"]])
    df = df.loc[~df["author"].isin(excluded)].copy()
    if lyon_only:
        df = df.loc[df["is_lyon"]].copy()
    if not include_replies:
        df = df.loc[df["is_reply"] == 0].copy()
    counts.append(("Dates, territoire, auteurs et réponses", len(df)))
    df["text_emb"] = (df["clean_text"].fillna("").map(str)
                      .str.replace(r"@[\w.-]+", " ", regex=True)
                      .str.replace("#", " ", regex=False)
                      .str.replace(r"\s+", " ", regex=True).str.strip())
    df = df.loc[df["text_emb"].str.split().str.len() >= min_words].copy()
    counts.append(("Longueur du texte sémantique", len(df)))
    # Sans likes/reposts : conserver le plus récent, puis départager par URI.
    df = df.sort_values(["created_at", "uri"], ascending=[False, True])
    df["_key"] = df["text_emb"].str.lower().str.replace(r"[^\w]+", " ", regex=True).str.strip()
    df = df.drop_duplicates("_key").copy()
    counts.append(("Dédoublonnage normalisé", len(df)))
    rank = df.groupby(["author", "date"]).cumcount()
    df = df.loc[rank < max_posts_per_author_day].copy()
    counts.append(("Plafond auteur / jour", len(df)))
    df["texte_clean"] = df["text"].map(lambda t: analysis_text(t, lyon_only))
    df = df.loc[df["texte_clean"].str.split().str.len() >= min_lexical_words].copy()
    counts.append(("Longueur du texte lexical", len(df)))
    df = df.drop(columns="_key").sort_values(["created_at", "uri"]).reset_index(drop=True)
    return df, pd.DataFrame(counts, columns=["Étape", "Posts"]), stats

class SourceAnalysisNotApplicable(ValueError):
    """The original settings cannot produce a usable result on this corpus."""


## TF-IDF / K-Means

Parametres et calculs de la source : min_df=5, max_df=0.5, unigrammes/bigrammes, 14 groupes par defaut, n_init=10 et graine 42. Silhouette euclidienne ; ARI avec n_init=3 et graines 1, 2, 3. Les vecteurs nuls sont inclus comme dans la source ; aucun ajustement silencieux du nombre de groupes.

In [ ]:
def fit_kmeans_topics(df, n_clusters=14, min_df=5, seed=SEED):
    df = df.copy()
    if len(df) <= n_clusters:
        raise ValueError("Le corpus doit contenir plus de posts que N_KMEANS ; aucun ajustement automatique.")
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score, adjusted_rand_score

    tfidf = TfidfVectorizer(min_df=min_df, max_df=0.5, ngram_range=(1, 2), sublinear_tf=True)
    Xt = tfidf.fit_transform(df["texte_clean"])
    terms = np.array(tfidf.get_feature_names_out())

    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=seed).fit(Xt)
    df["kmeans"] = km.labels_
    sil = silhouette_score(Xt, km.labels_, sample_size=min(5000, len(df)), random_state=seed)
    labs = [KMeans(n_clusters=n_clusters, n_init=3, random_state=s).fit(Xt).labels_ for s in (1, 2, 3)]
    ari = np.mean([adjusted_rand_score(km.labels_, l) for l in labs])

    rows = []
    for k in range(n_clusters):
        top_idx = km.cluster_centers_[k].argsort()[::-1][:8]
        rows.append({"kmeans": k, "posts": int((km.labels_ == k).sum()), "top_terms": ", ".join(terms[top_idx])})
    km_info = pd.DataFrame(rows).sort_values("posts", ascending=False)

    km_info["authors"] = km_info["kmeans"].map(df.groupby("kmeans")["author"].nunique()).fillna(0).astype(int)
    km_info["label"] = km_info["top_terms"].map(lambda value: " / ".join(value.split(", ")[:3]))
    km_info["examples"] = km_info["kmeans"].map(lambda k: " || ".join(df.loc[df.kmeans == k, "text_emb"].head(2).str[:180]))
    df["kmeans_label"] = df["kmeans"].map(km_info.set_index("kmeans")["label"])
    diagnostics = {"requested_k": n_clusters, "actual_k": int(df.kmeans.nunique()),
                   "features": Xt.shape[1], "fitted_posts": len(df),
                   "zero_vector_posts": int((Xt.getnnz(axis=1) == 0).sum()),
                   "zero_vectors_included": True,
                   "silhouette_euclidean": float(sil), "stability_ari": float(ari), "seed": seed}
    return df, km_info.reset_index(drop=True), diagnostics


## Embeddings et BERTopic

Le cache dépend du modèle, des URI **et des textes dans leur ordre**. Aucun pickle n’est chargé. BERTopic est une étape explicite : elle nécessite les dépendances optionnelles et le modèle local ou téléchargeable. Les étiquettes brutes et les réaffectations d’outliers sont conservées pour ne pas masquer le bruit.

In [ ]:
def cached_embeddings(df, directory, model_name=EMB_MODEL, model=None):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    payload = {"model": model_name, "rows": list(zip(df["uri"].astype(str), df["text_emb"].astype(str)))}
    fingerprint = hashlib.sha256(json.dumps(payload, ensure_ascii=False).encode("utf-8")).hexdigest()
    meta_path, emb_path = directory / "embeddings_metadata.json", directory / "embeddings.npy"
    if meta_path.exists() and emb_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if meta.get("fingerprint") == fingerprint:
            emb = np.load(emb_path, allow_pickle=False)
            if emb.ndim == 2 and emb.shape[0] == len(df) and emb.shape[1] > 0 and np.isfinite(emb).all():
                return emb, {"cache_hit": True, "fingerprint": fingerprint, "model": model_name}
    if not len(df):
        raise ValueError("Aucun texte pour les embeddings.")
    if model is None:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(model_name)
    emb = np.asarray(model.encode(df["text_emb"].tolist(), batch_size=64,
                                  show_progress_bar=True, normalize_embeddings=True))
    if emb.ndim != 2 or len(emb) != len(df) or not np.isfinite(emb).all():
        raise ValueError("Embeddings invalides.")
    np.save(emb_path, emb, allow_pickle=False)
    meta = {"cache_hit": False, "fingerprint": fingerprint, "model": model_name,
            "shape": list(emb.shape)}
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return emb, meta


def fit_bertopic_topics(df, embeddings, model, target_topics=30, reassign_outliers=True,
                        outlier_threshold=0.35, min_cluster_size=15, seed=SEED):
    from bertopic import BERTopic
    from bertopic.representation import MaximalMarginalRelevance
    from umap import UMAP
    from hdbscan import HDBSCAN
    if len(df) < max(10, min_cluster_size):
        raise ValueError("Corpus trop petit pour les paramètres BERTopic choisis.")
    if embeddings.ndim != 2 or len(embeddings) != len(df) or not np.isfinite(embeddings).all():
        raise ValueError("Les embeddings doivent correspondre exactement au corpus.")
    docs = df["text_emb"].tolist()
    vectorizer = CountVectorizer(stop_words=sorted(source_stopwords()[1]),
                                  ngram_range=(1, 2), min_df=5,
                                  token_pattern=r"(?u)\b[^\W\d_]{3,}\b")
    umap_model = UMAP(n_neighbors=15, n_components=5,
                      min_dist=0.0, metric="cosine", random_state=seed)
    clusterer = HDBSCAN(min_cluster_size=max(min_cluster_size, int(len(df)*0.002)), min_samples=5,
                        metric="euclidean", cluster_selection_method="eom", prediction_data=True)
    representation = MaximalMarginalRelevance(diversity=0.4)
    topic_model = BERTopic(embedding_model=model, umap_model=umap_model, hdbscan_model=clusterer,
                           vectorizer_model=vectorizer, representation_model=representation,
                           calculate_probabilities=False, verbose=False)
    try:
        labels, _ = topic_model.fit_transform(docs, embeddings=embeddings)
    except ValueError as error:
        if not any(message in str(error) for message in ("After pruning", "max_df corresponds", "empty vocabulary")):
            raise
        raise SourceAnalysisNotApplicable(
            "Source BERTopic settings (min_df=5): empty vocabulary during topic fitting; "
            "collect more data or explicitly change the source settings."
        ) from error
    found = set(labels) - {-1}
    if isinstance(target_topics, int) and target_topics >= 2 and len(found) > target_topics:
        topic_model.reduce_topics(docs, nr_topics=target_topics)
        labels = list(topic_model.topics_)
    original = np.asarray(labels, dtype=int)
    final = original.copy()
    if reassign_outliers and (original == -1).any() and (original >= 0).any():
        final = np.asarray(topic_model.reduce_outliers(docs, original.tolist(), strategy="embeddings",
                            embeddings=embeddings, threshold=outlier_threshold), dtype=int)
    try:
        topic_model.update_topics(docs, topics=final.tolist(), vectorizer_model=vectorizer,
                                  representation_model=representation)
    except ValueError as error:
        if not any(message in str(error) for message in ("After pruning", "max_df corresponds", "empty vocabulary")):
            raise
        raise SourceAnalysisNotApplicable(
            "Source BERTopic settings (min_df=5): empty vocabulary after topic update; "
            "collect more data or explicitly change the source settings."
        ) from error
    result = df.copy()
    result["topic_raw"] = original
    result["topic"] = final
    result["is_outlier_reassigned"] = (original == -1) & (final >= 0)
    rows = []
    for label in sorted(set(final)-{-1}):
        indices = np.flatnonzero(final == label)
        terms = [word for word, _ in (topic_model.get_topic(int(label)) or [])[:8]]
        centre = normalize(embeddings[indices].mean(axis=0, keepdims=True))[0]
        scores = normalize(embeddings[indices]) @ centre
        examples = df.iloc[indices[np.argsort(-scores)[:2]]]["text_emb"].str[:180].tolist()
        rows.append({"topic": int(label), "posts": len(indices), "authors": int(df.iloc[indices]["author"].nunique()),
                     "label": " / ".join(terms[:3]), "top_terms": ", ".join(terms),
                     "examples": " || ".join(examples)})
    info = pd.DataFrame(rows, columns=["topic","posts","authors","label","top_terms","examples"])
    if not info.empty:
        info = info.sort_values("posts", ascending=False).reset_index(drop=True)
    result["topic_label"] = result["topic"].map(info.set_index("topic")["label"]).fillna("hors-topic")
    diagnostics = {"topics": len(info), "raw_outlier_share": float((original == -1).mean()),
                   "final_outlier_share": float((final == -1).mean()),
                   "reassigned_posts": int(result["is_outlier_reassigned"].sum()),
                   "seed": seed, "outlier_reassignment_enabled": reassign_outliers}
    return result, info, diagnostics, topic_model


## Series et emergence

Calcul source : z = (observe - attendu) / sqrt(attendu + 1), reference de 28 jours et fenetre recente de 3 jours. La regle max(nombre de posts de reference, 1) est conservee, sans seuil de couverture ajoute. Le pic de chaque terme est selectionne avant verification des auteurs. Jaccard regroupe les 600 premiers termes sans separation par date ; moins de deux termes ne forme aucun evenement. Les historiques trop courts sont signales explicitement.

In [ ]:
EMERGENCE_COLUMNS = ["term", "peak_date", "z", "n_posts", "n_authors", "_docs"]
EVENT_COLUMNS = ["event", "label", "n_terms", "n_posts", "n_authors", "peak_date", "z_max", "examples", "main_topic", "_docs"]


def topic_time_series(df, topic_column="kmeans"):
    valid = df.loc[df[topic_column] >= 0].copy()
    if valid.empty:
        return pd.DataFrame(), pd.DataFrame()
    days = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    counts = valid.groupby(["date", topic_column]).size().unstack(fill_value=0).reindex(days, fill_value=0)
    # Comme la source : normalisation sur les posts affectés à un thème.
    totals = counts.sum(axis=1)
    shares = counts.div(totals.replace(0, np.nan), axis=0)
    return counts, shares

def detect_emerging_terms(df, window_days=3, baseline_days=28, z_min=4, min_docs=6,
                          min_authors=2, study_start=None, study_end=None):

    if df.empty:
        return pd.DataFrame(columns=EMERGENCE_COLUMNS), {"status": "empty_corpus", "evaluated_windows": 0}
    W, BASE, Z_MIN, MIN_DOCS, MIN_AUTHORS = window_days, baseline_days, z_min, min_docs, min_authors
    if min(W, BASE, MIN_DOCS, MIN_AUTHORS) < 1:
        raise ValueError("Les fenêtres et supports doivent être positifs.")
    COLLECT_START, END = pd.to_datetime(df["date"]).min().normalize(), pd.to_datetime(df["date"]).max().normalize()
    START = pd.Timestamp(study_start) if study_start is not None else COLLECT_START
    if study_end is not None:
        END = min(END, pd.Timestamp(study_end))
    diagnostics = {"status": "insufficient_history", "calendar_days": (END-COLLECT_START).days+1,
                   "required_calendar_days": BASE+W, "evaluated_windows": 0, "window_days": W,
                   "baseline_days": BASE, "z_min": Z_MIN, "min_docs": MIN_DOCS, "min_authors": MIN_AUTHORS}
    if diagnostics["calendar_days"] < BASE+W:
        return pd.DataFrame(columns=EMERGENCE_COLUMNS), diagnostics
    from scipy import sparse

    cv_e = CountVectorizer(binary=True, min_df=MIN_DOCS, ngram_range=(1, 2),
                           token_pattern=r"(?u)\b[a-z]{3,}\b")
    Xe = cv_e.fit_transform(df["texte_clean"]).tocsc()
    vocab = np.array(cv_e.get_feature_names_out())

    all_days = pd.date_range(COLLECT_START, END, freq="D")
    day_idx = df["date"].map({d: i for i, d in enumerate(all_days)}).values.astype(int)
    D = len(all_days)
    Dm = sparse.csr_matrix((np.ones(len(df)), (np.arange(len(df)), day_idx)), shape=(len(df), D))
    C = (Xe.T @ Dm).toarray()                        # termes x jours (nb de posts contenant le terme)
    N = np.bincount(day_idx, minlength=D).astype(float)

    Z = np.full_like(C, -np.inf, dtype=float)
    OBS = np.zeros_like(C, dtype=float)
    for d in range(BASE + W - 1, D):
        b0, b1 = d - W - BASE + 1, d - W + 1          # jours de baseline : b0 .. d-W
        base_rate = C[:, b0:b1].sum(1) / max(N[b0:b1].sum(), 1)
        obs = C[:, d - W + 1:d + 1].sum(1)
        exp = base_rate * N[d - W + 1:d + 1].sum()
        Z[:, d] = (obs - exp) / np.sqrt(exp + 1)
        OBS[:, d] = obs

    study_days = np.where((all_days >= START) & (all_days <= END))[0]
    cands = []
    for t in range(len(vocab)):
        zs = np.where(OBS[t, study_days] >= MIN_DOCS, Z[t, study_days], -np.inf)
        j = int(np.argmax(zs))
        if zs[j] >= Z_MIN:
            d = study_days[j]
            docs_in = np.where((day_idx > d - W) & (day_idx <= d))[0]
            docs_in = docs_in[Xe[docs_in, t].toarray().ravel() > 0]
            n_auth = df["author"].iloc[docs_in].nunique()
            if n_auth >= MIN_AUTHORS:
                cands.append({"term": vocab[t], "peak_date": all_days[d], "z": zs[j],
                              "n_posts": len(docs_in), "n_authors": n_auth, "_docs": set(docs_in)})
    emerg = pd.DataFrame(cands, columns=EMERGENCE_COLUMNS).sort_values("z", ascending=False).reset_index(drop=True)

    diagnostics["status"] = "ok"
    diagnostics["evaluated_windows"] = max(0, D-(BASE+W-1))
    diagnostics["source_zero_baseline_rule"] = "max(reference_posts, 1), without coverage threshold"
    return emerg, diagnostics


def group_emerging_events(terms, df, distance_threshold=0.8, max_terms=600):
    emerg = terms
    from sklearn.cluster import AgglomerativeClustering

    def jaccard_matrix(sets):
        n = len(sets); M = np.zeros((n, n))
        for i in range(n):
            for j in range(i, n):
                u = len(sets[i] | sets[j])
                M[i, j] = M[j, i] = (len(sets[i] & sets[j]) / u) if u else 0
        return M

    evt_rows, events = [], pd.DataFrame(columns=EVENT_COLUMNS)
    if len(emerg) >= 2:
        sub = emerg.head(max_terms)                                      # borne le coût de la matrice
        sets = sub["_docs"].tolist()
        dist = 1 - jaccard_matrix(sets)
        try:
            agg = AgglomerativeClustering(n_clusters=None, metric="precomputed", linkage="average",
                                          distance_threshold=distance_threshold)
        except TypeError:                                          # sklearn < 1.2
            agg = AgglomerativeClustering(n_clusters=None, affinity="precomputed", linkage="average",
                                          distance_threshold=distance_threshold)
        sub = sub.assign(event=agg.fit_predict(dist))

        for ev, g in sub.groupby("event"):
            g = g.sort_values("z", ascending=False)
            union = sorted(set().union(*g["_docs"]))
            examples = df.iloc[union]
            if "likes" in examples:
                examples = examples.nlargest(2, "likes")
            else:
                examples = examples.head(2)
            ex = examples["text_emb"].str[:140].tolist()
            topic_column = "topic_label" if "topic_label" in df else "kmeans_label"
            main_topic = df.iloc[union][topic_column].mode().iloc[0] if topic_column in df else None
            evt_rows.append({
                "event": ev, "label": ", ".join(g["term"].head(5)), "n_terms": len(g),
                "n_posts": len(union), "n_authors": df["author"].iloc[union].nunique(),
                "peak_date": g["peak_date"].iloc[0].date(), "z_max": round(g["z"].iloc[0], 1),
                "main_topic": main_topic,
                "examples": " || ".join(ex), "_docs": set(union),
            })
        events = pd.DataFrame(evt_rows).sort_values("z_max", ascending=False).reset_index(drop=True)
    else:
        pass

    return events


## Comparaison aux médias

On utilise le fichier réel `lyon-news-mining/data/processed/article_topics.parquet` et les colonnes `km`, `km_label`, `title_clean` du projet presse. Les numéros de clusters ne sont jamais assimilés entre sources. Une comparaison sémantique nécessite le même modèle d’embeddings pour les deux corpus et un chevauchement temporel. Une corrélation n’établit ni causalité ni antériorité journalistique.

In [ ]:
def load_media_corpus(topics_path, corpus_path=None):
    """Join media topics and article titles by article_id when needed."""
    topics_path = Path(topics_path)
    media = pd.read_parquet(topics_path)
    if not ({"title_clean", "title"} & set(media.columns)):
        corpus_path = Path(corpus_path) if corpus_path is not None else topics_path.with_name("corpus.parquet")
        if not corpus_path.exists():
            raise FileNotFoundError("Titles require corpus.parquet next to article_topics.parquet.")
        corpus = pd.read_parquet(corpus_path)
        if "article_id" not in media or "article_id" not in corpus:
            raise ValueError("Media join requires article_id in both files.")
        additions = [column for column in ("title_clean", "title", "in_analysis", "date") if column in corpus and column not in media]
        media = media.merge(corpus[["article_id"] + additions], on="article_id", how="inner", validate="one_to_one")
    return media


def cosine_topic_centroids(embeddings, labels):
    labels = np.asarray(labels)
    keys = sorted(set(labels) - {-1})
    if not keys:
        return [], np.empty((0, embeddings.shape[1]))
    centres = np.vstack([embeddings[labels == key].mean(axis=0) for key in keys])
    return keys, normalize(centres)


def compare_media_topics(bluesky, media, encoder, topic_column="kmeans", min_overlap_days=2,
                         similarity_threshold=0.45, max_lag=7):
    columns = ["topic_bluesky","theme_media","similarity","posts_bluesky","articles_media",
               "correlation_same_day","best_lag_days","best_lag_correlation","matched"]
    if "km" not in media.columns or not ({"title_clean","title"} & set(media.columns)) or "date" not in media:
        raise ValueError("Colonnes médias requises : date, km, title_clean (ou title).")
    b, m = bluesky.copy(), media.copy()
    b["date"] = pd.to_datetime(b["date"]).dt.normalize()
    m["date"] = pd.to_datetime(m["date"], utc=True).dt.tz_localize(None).dt.normalize()
    if "in_analysis" in m:
        m = m.loc[m["in_analysis"]].copy()
    if b.empty or m.empty:
        return pd.DataFrame(columns=columns), {"status":"empty_corpus"}
    start, end = max(b["date"].min(),m["date"].min()), min(b["date"].max(),m["date"].max())
    if start>end:
        return pd.DataFrame(columns=columns), {"status":"no_date_overlap"}
    b = b.loc[b["date"].between(start,end) & (b[topic_column]>=0)].reset_index(drop=True)
    m = m.loc[m["date"].between(start,end)].dropna(subset=["km"]).reset_index(drop=True)
    if b.empty or m.empty:
        return pd.DataFrame(columns=columns), {"status":"empty_overlap"}
    text_column = "title_clean" if "title_clean" in m else "title"
    b_emb = np.asarray(encoder.encode(b["text_emb"].tolist(), normalize_embeddings=True, show_progress_bar=False))
    m_emb = np.asarray(encoder.encode(m[text_column].fillna("").astype(str).tolist(), normalize_embeddings=True, show_progress_bar=False))
    b_keys,b_centres = cosine_topic_centroids(b_emb,b[topic_column])
    m_keys,m_centres = cosine_topic_centroids(m_emb,m["km"])
    similarities = b_centres @ m_centres.T
    days = pd.date_range(start,end)
    bc = b.groupby(["date",topic_column]).size().unstack(fill_value=0).reindex(days,fill_value=0)
    mc = m.groupby(["date","km"]).size().unstack(fill_value=0).reindex(days,fill_value=0)
    bs = bc.div(bc.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)
    ms = mc.div(mc.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)
    rows=[]
    for i,key in enumerate(b_keys):
        j=int(similarities[i].argmax()); other=m_keys[j]
        a, z = bs[key].rolling(3, center=True, min_periods=1).mean(), ms[other].rolling(3, center=True, min_periods=1).mean()
        correlations={}
        for lag in range(-max_lag,max_lag+1):
            aligned=pd.concat([a,z.shift(-lag)],axis=1).dropna()
            if len(aligned)>=min_overlap_days and (aligned.std()>0).all():
                correlations[lag]=float(aligned.iloc[:,0].corr(aligned.iloc[:,1]))
        best=max(correlations,key=correlations.get) if correlations else None
        rows.append({"topic_bluesky":int(key),"theme_media":int(other),"similarity":float(similarities[i,j]),
                     "posts_bluesky":int(bc[key].sum()),"articles_media":int(mc[other].sum()),
                     "correlation_same_day":correlations.get(0), "best_lag_days":best,
                     "best_lag_correlation":correlations.get(best), "matched":bool(similarities[i,j]>=similarity_threshold)})
    return pd.DataFrame(rows,columns=columns), {"status":"ok","overlap_calendar_days":len(days),
              "lag_interpretation":"positive: Bluesky precedes media; descriptive only",
              "minimum_aligned_observations":min_overlap_days}
